In [1]:
# imports

# show rich formats 
from IPython.display import display, Markdown

# get python to interact with openai
from openai import OpenAI

# use personal openai key
import os
from dotenv import load_dotenv

# check load_dotenv works - should come back 'True'
# load_dotenv()

# make a destination 'resumes' directory for the work

os.makedirs("resumes", exist_ok=True)

# use python to format markdown to html
from markdown import markdown

#### Because we're sending this out as a resume, we want it to be in a .pdf format and that means we'll have to use a library called <span style="color:green">Weasy</span>... 
#### <i><b><span style="color:red">Whoop!</span></i></b>
#### Nope. Sorry. 
#### Not using weasyprint. Lining Weasyprint libraries up with each particular <span style="color:blue">Python</span> environment and setting / dedicating kernals still causes nightmares of 'public-speaking-in-underpants' proportions. 
#### Surely a powerful tool, but it's got the playfulness and ease of use of a rabid porcupine. <i><span style="color:firebrick">No grazie</span></i>.
#### Instead, going with PDFKIT. Working in HTML, so it'll do the job.
##### * Note: PDFKIT works using wkhtmltopdf, which <i>in very simple terms</i> converts a webpage to a pdf file. Please see [reference.txt](https://github.com/npj210mlk/jobapp_prompter/blob/main/requirements.txt) in the repo for installation instructions.


In [2]:
# import pdfkit

# # test
# pdfkit.from_string("<h1>It should be called 'QueezyPrint,' amirite?</h1>", "output.pdf")

#### ha! apparently, jupyter agrees - came back 'True'

***
#### with the libraries imported we can now break the project down into four (4) steps:
##### 1.) open and read the markdown resume file
#####     * see requirements notes
##### 2.) input the desired job description
##### 3.) template some prompt engineering
##### 4.) convert markdown to html
##### 5.) convert html to pdf
***
*** 
#### Step 1: Open and Read the Markdown Resume
****

In [3]:
# FOR PERSONAL USE
# open resume and read it
with open("/Users/nicholasjoseph/Desktop/nj_master_resume.md", "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

# # FOR REPO USE:
# with open("/Users/nicholasjoseph/Desktop/nj_safe_master_resume.md", "r", encoding = "utf-8") as resume_file:
#     resume_string = resume_file.read()

# check to see if it worked:
# display(Markdown(resume_string))

# markdown resume displays as expected.

***
#### Step 2: Input the Desired Job Description
***

In [5]:
# job description will be from anywhere, so input is used to pause the program until we find one to copy/paste into this variable 
job_desc_string = input()

 Description    Job Summary:  As a Data Engineer, you will spearhead the evolution of data pipelines—bridging the gap between legacy systems and modern cloud-based environments. You will implement our client’s data mart and customer analytics platform, and integrate data spanning on-premises Microsoft SQL Server and Google Cloud Platform services.  Duties/Responsibilities:  Technical Solutions Architecture: Design and construct the technical architecture to support the solution. Define and implement architectural components, and develop data requirements, data maps, transformation rules, and aggregation plans. Define strategies for data extraction, hygiene, acquisition, and loading. Design & Build: Build scalable ETL pipelines utilizing Python, SQL Server, and SSIS alongside dbt and Airflow. Develop and optimize data warehousing solutions. Cloud Integration & Modernization: Oversee migration projects from legacy systems to GCP, and integrate and harmonize data across hybrid environment

***
#### Step 3: Template Some Prompt Engineering
##### - this involves dealing with ChatGPT, particularly the ChatGPT 4o-mini pre-trained transformer.
##### <u>a couple of things about the ChatGPT 4o-mini engine (model)</u>:
#####     a.) it is the smaller, more affordable version of the massive GPT-4o engine available to developers; and
#####     b.) because its focus is on text classificaton / prediction, it is purely a "decoder-only" type transformer
#### The idea is to have ChatGPT create the prompt to respond to the likely Applicant Tracking System being used by the job poster.
##### Reason being, machines talk to machines better than we can. 
***

In [6]:
# making up a random "lambda" function to create the variable "prompt_template"

# the text is what I would be putting into ChatGPT were I to do this one job at a time - prompt engineering

prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts
    - Select only the core competencies most relevant to the job description

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Summary Selection**:
    - Please tailor the best best Summary format for the job description and Recruiter expectations.

4. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

5. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.

6. **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

In [7]:
# set the prompt variable for our ChatGPT message roles
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 4: Generate the Resume with GPT-4o-mini
##### - Make the api call and tell GPT to write the resume using the prompt from above
***

In [8]:
# set up api client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

# make the call

# set response variable to hold the all info we get back
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    # set our roles up - think of casting a play: "Today, the role of the Expert Resume Writer will be played by the system."
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
    
    # give it some creative license 
    temperature = 0.7
)

# get our response
response_string = response.choices[0].message.content

***
#### Step 5: Show Us the Results
***

In [9]:
# separate new resume from improvements AI suggests at 'Additional Suggestions'
response_list = response_string.split("## Additional Suggestions")

In [10]:
display(Markdown(response_list[0]))

# Nicholas Joseph
**Data Engineer | Product Manager | Tech Enthusiast**

Email: [nickpjoseph210@gmail.com](mailto:nickpjoseph210@gmail.com) | Phone: (210) 771-9853 | LinkedIn: [nickjoseph](https://www.linkedin.com/in/nickjoseph) | [GitHub](https://github.com/npj210mlk)

---

### Career Summary  

Results-oriented Data Engineer with 5+ years of experience in designing, constructing, and optimizing data pipelines that bridge legacy systems and modern cloud environments. Proficient in Python, SQL Server, and ETL processes, with a strong focus on enhancing data accessibility and quality. Proven ability to collaborate with cross-functional teams and stakeholders to deliver strategic data solutions that drive business insights and performance improvements.

---

### Relevant Experience

**Cloud Data Engineer**  
**Data Clymer (now Spaulding Ridge)**, Remote | 07/2022 - 06/2023  
- Designed and built scalable ETL pipelines for high-profile initiatives, reducing data extraction times by over 98.9% and improving reporting accuracy by 53%.  
- Collaborated with Data Architects and stakeholders to implement data integration strategies across hybrid environments, ensuring alignment with business requirements.  
- Conducted knowledge transfer sessions, enhancing team capabilities in data pipeline architecture and cloud integration.

**Data Engineer**  
**Gathi Analytics (now Apexon)**, Remote | 12/2020 - 03/2022  
- Led migration of legacy banking data to cloud-based architectures (e.g., Snowflake), improving accessibility and compliance for regulatory reporting.  
- Developed interactive dashboards and comprehensive documentation, enforcing data quality standards and best practices.  
- Mentored junior team members through the Gathi Garden program, fostering talent development in data engineering.

**Project Lead**  
**Noble Services, Inc.**, San Antonio, TX | 10/2017 - 01/2020  
- Architected and engineered scalable data solutions, integrating APIs and advanced machine learning models for enhanced client applications.  
- Managed diverse teams to deliver projects on time and within budget, achieving five-star ratings across customer platforms.

---

### Skills  

**Technical Skills:**  
- **Languages & Tools:** Python, SQL Server (T-SQL, SSIS), dbt, Airflow, Google Cloud Platform, Snowflake  
- **Data Engineering:** ETL/ELT processes, Data Warehousing, Data Modeling, Cloud Integration  
- **Project Management:** Agile Methodologies, Stakeholder Management, Documentation  

**Soft Skills:**  
- Effective Communication, Cross-Functional Collaboration, Problem-Solving, Mentoring, Conflict Resolution

---

### Education  

**Certificate of Completion** - PMP Exam Prep - 35 PDU  
Technical Institute of America, Remote | 2024  

**Certificate of Completion** - Data Science  
Codeup, San Antonio, TX | 2020  

**Bachelor's** - Biology  
Concordia University, Austin, TX | 2001  

---

#

***
#### Step 6: Save the New Resume
##### Great. Hit several snags. You either need WeasyPrint installed in some capacity, or other tools I found (like 'Grip') are difficult to automate and test on MacOS. 
##### Looks like the programming gods have spoken: we're going with WeasyPrint.
##### <center><span style ="color:red"><b><u>To Do This:</u></b></span><center>
##### 1.) completely uninstall existing WeasyPrint's existence from your machine with pip and brew uninstalls;
##### 2.) run a 'brew cleanup' just to wipe out any remnants;
##### 3.) form home directory in Terminal (I'm using Mac), type 'brew install cairo pango gdk-pixbuf libffi' - these are the native Weasyprint dependencies;
##### 4.) set your environment variables in your profile (for me, my ~/.zshrc file) to the following:
###### export DYLD_LIBRARY_PATH=/opt/homebrew/lib:$DYLD_LIBRARY_PATH
###### export LIBRARY_PATH="/opt/homebrew/lib:$LIBRARY_PATH"
###### export PKG_CONFIG_PATH="/opt/homebrew/lib/pkgconfig:/opt/homebrew/share/pkgconfig"
###### export PATH="/opt/homebrew/bin:$PATH"
##### 5.) restart the terminal;
##### 6.) navigate to your project folder;
##### 7.) type 'pip install weasyprint markdown'; and (finally)
##### 8.) open your notebook from your project file with 'jupyter notebook'
***

In [11]:
# Markdown was already imported earlier
# import WeasyPrint and its HTML abilities
from weasyprint import HTML


# # SAVE AS PDF FOR REPO:
# output_pdf_file = "resumes/nick_joseph_resume.pdf"

# SAVE AS PDF FOR PERSONAL USE:
# output_pdf_file = "/Users/nicholasjoseph/Desktop/nick_joseph_resume.pdf"



# convert the Markdown file to HTML
html_content = markdown(response_list[0])

# now take that HTML and convert it into a PDF and save
HTML(string=html_content).write_pdf(output_pdf_file)

In [12]:
# # let's save the new Markdown file for REPO, too
# markdown_output = "resumes/nickjoseph_chatgpt_resume.md"

# SAVE AS MARKDOWN FOR PERSONAL USE:
markdown_output = "/Users/nicholasjoseph/Desktop/nickjoseph_chatgpt_resume.md"

with open(markdown_output, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement
##### Because we split "response_string" earlier, it was split at "Additional Suggestions," giving two items "response_list."
##### The first item ([0]) is the resume ChatGPT wrote with our prompt.
##### The second item ([1]) are the suggestions ChatGPT offers to make our resume stronger
***

In [13]:
# show us how to make the resume stronger
display(Markdown(response_list[1]))



1. **Skills to Strengthen Alignment:**
   - **Cloud Pipeline Orchestration**: Familiarize yourself with Google Cloud Composer.
   - **Data Quality Tools**: Explore tools for data quality management (e.g., Great Expectations).
  
2. **Certifications/Courses to Pursue:**
   - **Google Cloud Professional Data Engineer** certification.
   - **Advanced Python for Data Engineering** course.

3. **Project Ideas or Experiences:**
   - Develop a personal project focusing on migrating a dataset from a legacy system to a cloud-based platform utilizing ETL processes.
   - Collaborate on open-source projects that emphasize data pipeline construction and cloud integration for real-world applications.

By following these suggestions and utilizing the tailored resume, you'll enhance your chances of aligning closely with the Data Engineer role you're pursuing.

In [13]:
# # markdown is already imported
# from weasyprint import HTML

In [14]:
# from markdown2pdf import convert

# markdown_resume = display(Markdown(response_list[0]))

# convert(markdown_resume, "nj_resume_in_pdf.pdf")